# Proyectos detrás de las variaciones · Consulta Amigable (MEF) · Loreto, San Martín y Ucayali

Complementa el archivo **Variaciones alarmantes** de `Fiscal_Procesamiento.ipynb`. Para cada departamento y nivel de
gobierno (G.N., G.R., G.L.) compara la inversión (genérica 2.6) del último mes con el mismo mes del año anterior:

| Total del nivel | Funciones que se abren | Año que se consulta |
|---|---|---|
| Diferencia positiva | las 5 de **▲ Mayores aumentos** | año actual (p. ej. Set-26) |
| Diferencia negativa | las 5 de **▼ Mayores caídas** | año anterior (p. ej. Set-25) |

En cada función baja a **Producto/Proyecto** (Genérica 6-26 > Mes > Departamento (Meta) > Nivel > Función > Proyecto)
y guarda los **5 proyectos con mayor devengado** de ese mes. No cruza proyectos entre años: si un proyecto del año
anterior ya no figura en el actual, aparece igual en la consulta del año anterior, que es la que explica la caída.

- **Entrada:** los CSV `DEV Funcion.*` que deja `Fiscal_Scraper.ipynb` en `Fiscal/Scrapeo/output/DEVENGADO/` (con ellos se
  eligen las funciones, igual que en Variaciones alarmantes). Al MEF solo se le consultan los proyectos (~45 consultas).
- **Salida:** `Proyectos variaciones AAAA-MM.xlsx` en la carpeta de Drive `PROYECTOS_FOLDER_ID`: una hoja por
  departamento y una hoja `DETALLE` con filtro.
- 16: Comunicaciones se suma a 15: Transporte (como en los cuadros): se consultan ambas y se juntan sus proyectos.

In [ ]:
# ═══ BLOQUE 1 · LIBRERÍAS Y DRIVE ═══
!pip install requests beautifulsoup4 pandas numpy openpyxl --quiet
import os, re, glob, time, random, unicodedata, warnings
from datetime import date, datetime
import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter as L
from IPython.display import display
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")
RUTA_FISCAL = "/content/drive/MyDrive/Fiscal"
assert os.path.isdir(RUTA_FISCAL), f"No encuentro {RUTA_FISCAL}: revisa el nombre de la carpeta en Drive"
print("✅ Librerías listas · Drive montado")

In [ ]:
# ═══ BLOQUE 2 · CONFIGURACIÓN ═══
RUTA_SCRAPEO = os.path.join(RUTA_FISCAL, "Scrapeo", "output")
RUTA_SALIDA  = os.path.join(RUTA_FISCAL, "Procesamiento", "output")    # copia de respaldo si falla la subida
PROYECTOS_FOLDER_ID = "1aUFb4bqz3ZU6_5bbTRYl4f0V4-nX2Cy2"   # carpeta de Drive donde se crea "Proyectos variaciones AAAA-MM.xlsx"
PROYECTOS_NOMBRE = "Proyectos variaciones"

MES_ANALISIS = None            # None = último mes con datos en los CSV · o "2026-09"
TOP_FUNCIONES = 5              # funciones que se abren por nivel (las mismas del cuadro de Variaciones alarmantes)
TOP_PROYECTOS = 5              # proyectos por función (los de mayor devengado del mes)
AGRUPAR_COMUNICACIONES_EN_TRANSPORTE = True   # 16: Comunicaciones se suma en 15: Transporte, como en los cuadros
PAUSA = (0.4, 1.0)             # segundos aleatorios entre consultas (Incapsula: pocas consultas y sin apuro)

DEPARTAMENTOS = {"16": "LORETO", "22": "SAN MARTÍN", "25": "UCAYALI"}
NIVELES = [  # código MEF, CSV del scraper, nombre en los cuadros, claves de departamento en la URL (como el scraper)
    dict(codigo="E", corto="GobNacional",   carpeta="Gobierno Nacional",  nombre="GOBIERNO NACIONAL", claves_depto=["21"]),
    dict(codigo="R", corto="GobRegional",   carpeta="Gobierno Regional",  nombre="GOBIERNO REGIONAL", claves_depto=["21"]),
    dict(codigo="M", corto="Municipalidad", carpeta="Gobierno Municipal", nombre="GOBIERNOS LOCALES", claves_depto=["5", "21"]),
]
FUNCIONES = {f"{i:02d}": n for i, n in enumerate([
    "LEGISLATIVA", "RELACIONES EXTERIORES", "PLANEAMIENTO, GESTION Y RESERVA DE CONTINGENCIA",
    "DEFENSA Y SEGURIDAD NACIONAL", "ORDEN PUBLICO Y SEGURIDAD", "JUSTICIA", "TRABAJO", "COMERCIO", "TURISMO",
    "AGROPECUARIA", "PESCA", "ENERGIA", "MINERIA", "INDUSTRIA", "TRANSPORTE", "COMUNICACIONES", "AMBIENTE",
    "SANEAMIENTO", "VIVIENDA Y DESARROLLO URBANO", "SALUD", "CULTURA Y DEPORTE", "EDUCACION",
    "PROTECCION SOCIAL", "PREVISION SOCIAL", "DEUDA PUBLICA"], start=1)}
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Set", "Oct", "Nov", "Dic"]

# Claves numéricas de la URL de Consulta Amigable (CONTEXTO_PROYECTO §4.2): 24 genérica, 23 mes, 1 nivel,
# 5 departamento, 21 departamento (Meta), 8 función, 31 proyecto. La PRIMERA clave vacía es la columna que se lista.
CLAVE_GENERICA, CLAVE_MES, CLAVE_FUNCION, CLAVE_PROYECTO = "24", "23", "8", "31"
GENERICA_INVERSION = "6-2-6"   # se verifica al calibrar
PARAMS_EXTRA = {}              # si F12 > Network muestra otros parámetros, se sobreescriben aquí

def etq_mes(f):
    return f"{MESES_ES[f.month - 1]}-{f:%y}"

def cod_agrupado(cod):
    return "15" if AGRUPAR_COMUNICACIONES_EN_TRANSPORTE and cod == "16" else cod

def nombre_funcion(cod):
    if AGRUPAR_COMUNICACIONES_EN_TRANSPORTE and cod == "15":
        return "15: TRANSPORTE (incl. 16: Comunicaciones)"
    return f"{cod}: {FUNCIONES.get(cod, '')}"

In [ ]:
# ═══ BLOQUE 3 · FUNCIONES A ABRIR (desde los CSV del scraper, mismo criterio que Variaciones alarmantes) ═══
_RE_MES = re.compile(r"(\d{4}) - Mes (\d{2}) - ")

def leer_funcion(n):
    partes = []
    for a in sorted(glob.glob(os.path.join(RUTA_SCRAPEO, "DEVENGADO", n["carpeta"], f"DEV Funcion.{n['corto']} * - Mes * - *.csv"))):
        m = _RE_MES.search(os.path.basename(a))
        if not m:
            continue
        df = pd.read_csv(a, encoding="utf-8-sig")
        df["FECHA"] = pd.Timestamp(int(m.group(1)), int(m.group(2)), 1)
        partes.append(df)
    if not partes:
        return pd.DataFrame()
    df = pd.concat(partes, ignore_index=True)
    df["COD_DEPTO"] = df["DEPARTAMENTO"].astype(str).str.extract(r"^(\d+)")[0]
    df["COD_FUNCION"] = df["FUNCION"].astype(str).str.extract(r"^(\d+)\s*:")[0].map(cod_agrupado)
    return df.groupby(["COD_DEPTO", "FECHA", "COD_FUNCION"], as_index=False)["DEVENGADO"].sum()

dev = {n["codigo"]: leer_funcion(n) for n in NIVELES}
if all(d.empty for d in dev.values()):
    raise FileNotFoundError(f"No hay CSV DEV Funcion.* en {os.path.join(RUTA_SCRAPEO, 'DEVENGADO')}: corre antes Fiscal_Scraper.")
u = pd.Timestamp(MES_ANALISIS + "-01") if MES_ANALISIS else max(d["FECHA"].max() for d in dev.values() if not d.empty)
u_ant = u - pd.DateOffset(years=1)

def tops(t):
    """Igual que Variaciones alarmantes: siempre TOP aumentos (mayores diferencias) y TOP caídas (menores), sin los
    ítems en 0 en ambos periodos y sin repetir un ítem en las dos listas."""
    t = t[(t["ANTERIOR"] != 0) | (t["ACTUAL"] != 0)]
    sube = t.nlargest(TOP_FUNCIONES, "VARIACION")
    baja = t.drop(sube.index).nsmallest(TOP_FUNCIONES, "VARIACION")
    return sube, baja

plan, avisos = [], []          # plan: una entrada por departamento y nivel
for cod, nombre in DEPARTAMENTOS.items():
    for n in NIVELES:
        d = dev[n["codigo"]]
        fechas = set(d["FECHA"]) if not d.empty else set()
        if u not in fechas or u_ant not in fechas:
            avisos.append(f"{nombre} · {n['nombre']}: faltan los CSV de {etq_mes(u_ant)} o {etq_mes(u)} (no se analiza).")
            continue
        s = d[d["COD_DEPTO"] == cod]
        a = s[s["FECHA"] == u_ant].groupby("COD_FUNCION")["DEVENGADO"].sum().rename("ANTERIOR")
        b = s[s["FECHA"] == u].groupby("COD_FUNCION")["DEVENGADO"].sum().rename("ACTUAL")
        t = pd.concat([a, b], axis=1).fillna(0.0)
        t["VARIACION"] = t["ACTUAL"] - t["ANTERIOR"]
        dif_total = float(t["VARIACION"].sum())
        positivo = dif_total >= 0
        sube, baja = tops(t)
        elegidas = sube if positivo else baja
        plan.append(dict(cod_depto=cod, departamento=nombre, nivel=n, total_anterior=float(t["ANTERIOR"].sum()),
                         total_actual=float(t["ACTUAL"].sum()), diferencia=dif_total, positivo=positivo,
                         fecha=u if positivo else u_ant,
                         funciones=[dict(cod=c, nombre=nombre_funcion(c), anterior=x["ANTERIOR"], actual=x["ACTUAL"],
                                         diferencia=x["VARIACION"]) for c, x in elegidas.iterrows()]))

print(f"Mes analizado: {etq_mes(u)} vs {etq_mes(u_ant)}")
display(pd.DataFrame([{"DEPARTAMENTO": p["departamento"], "NIVEL": p["nivel"]["nombre"],
                       "DIFERENCIA S/": round(p["diferencia"]), "SE CONSULTA": etq_mes(p["fecha"]),
                       "FUNCIONES": " · ".join(f["nombre"][:2] for f in p["funciones"])} for p in plan]))
for a in avisos:
    print("⚠️", a)

In [ ]:
# ═══ BLOQUE 4 · NAVEGACIÓN EN CONSULTA AMIGABLE (misma lógica que Fiscal_Scraper) ═══
URL_BASE = "https://apps5.mineco.gob.pe/transparencia/mensual/Navegar_6.aspx"
_sesion = None
def sesion():
    global _sesion
    if _sesion is None:
        _sesion = requests.Session()
        _sesion.headers.update({"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
                                "Referer": "https://apps5.mineco.gob.pe/transparencia/mensual/default.aspx"})
    return _sesion

FORMATOS = [
    {"nombre": "navegador GET", "metodo": "GET", "pre": {"_tgt": "frame", "_uhc": "yes"}, "post": {"ap": "ActProy"}},
    {"nombre": "GET + ap",      "metodo": "GET", "pre": {}, "post": {"ap": "ActProy"}},
    {"nombre": "POST + ap",     "metodo": "POST","pre": {}, "post": {"ap": "ActProy"}},
    {"nombre": "POST original", "metodo": "POST","pre": {}, "post": {}},
]
FORMATO = None
NIVEL_DE = {n["codigo"]: n for n in NIVELES}

def a_numero(txt):
    t = (txt or "").replace(",", "").replace("\xa0", "").strip()
    if t in ("", "-"):
        return 0.0
    try:
        return float(t)
    except ValueError:
        return np.nan

def normalizar(txt):
    t = unicodedata.normalize("NFKD", txt or "")
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

_FILTRO = re.compile(r"^(total|gen[eé]rica|nivel de gobierno|gob\.?\s?loc|departamento|provincia|distrito|"
                     r"funci[oó]n|mes\b|trimestre|sector|pliego|ejecutora|unidad ejecutora|fuente|rubro|categor|"
                     r"producto|divisi[oó]n|municipalidad|grupo funcional|sub-?gen[eé]rica|espec[ií]fica)", re.I)
_CODIGO = re.compile(r"^([\w\-\.]+)\s*:")

class Pagina:
    """filtros: breadcrumb [(etiqueta, devengado)] · filas: detalle [(valor_url, etiqueta, devengado)]"""
    def __init__(self, html, url):
        self.url = url
        s = BeautifulSoup(html, "html.parser")
        self.tiene_tabla = s.find("table", class_="Data") is not None
        self.filtros, self.filas = [], []
        for tr in s.find_all("tr"):
            if tr.find("table"):
                continue
            celdas = tr.find_all("td")
            if len(celdas) < 8:
                continue                                   # las filas "Sistema de Seguimiento de Inversiones SSI" y "Buscar ítem" no traen montos
            dev = a_numero(celdas[-3].get_text(strip=True))   # … Devengado | Girado | Avance %
            if dev != dev:
                continue
            textos = [" ".join(c.get_text(" ", strip=True).split()) for c in celdas[:3]]
            etiqueta = next((t for t in textos if t), "")
            radio = tr.find("input", attrs={"type": "radio"})
            if _FILTRO.match(etiqueta) and radio is None:
                self.filtros.append((etiqueta, dev))
                continue
            m = _CODIGO.match(etiqueta)
            valor = (radio.get("value") if radio is not None else None) or (m.group(1) if m else None)
            if valor is None:
                continue
            self.filas.append((valor.split("/")[0].strip(), etiqueta, dev))

def armar_params(anio, mes, nivel, depto, ruta, agrupar, formato):
    p = dict(formato["pre"]); p["0"] = ""
    p[CLAVE_MES] = mes
    p["1"] = nivel
    for k in NIVEL_DE[nivel]["claves_depto"]:
        p[k] = depto
    for k, v in ruta:
        p[k] = v
    p[agrupar] = ""
    p["y"] = anio
    p.update(formato["post"]); p.update(PARAMS_EXTRA)
    return p

def consultar(anio, mes, nivel, depto, ruta, agrupar, formato=None, reintentos=4):
    formato = formato or FORMATO
    params = armar_params(anio, mes, nivel, depto, ruta, agrupar, formato)
    ultimo = ""
    for i in range(reintentos):
        try:
            r = sesion().request(formato["metodo"], URL_BASE, params=params, timeout=40)
            if r.status_code == 200:
                p = Pagina(r.content, r.url)
                if p.tiene_tabla:
                    time.sleep(random.uniform(*PAUSA))
                    return p, None
                ultimo = "SIN_TABLA"
            else:
                ultimo = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            ultimo = f"{type(e).__name__}: {e}"
        time.sleep(min(30, 2 ** i + random.random()))
    return None, ultimo

def validar(p, nivel, depto, cod_funcion):
    """El breadcrumb debe confirmar nivel, departamento y función (si un filtro no se aplicó, la tabla sería otra)."""
    etiquetas = [normalizar(e) for e, _ in p.filtros]
    errores = []
    if not any(e.startswith("nivel de gobierno") and f" {nivel.lower()}:" in e for e in etiquetas):
        errores.append(f"la página no está filtrada por nivel {nivel}")
    if not any(e.startswith("departamento") and f"{depto}:" in e for e in etiquetas):
        errores.append(f"la página no corresponde al departamento {depto}")
    if cod_funcion and not any(e.startswith("funci") and f" {cod_funcion}:" in e for e in etiquetas):
        errores.append(f"la página no está filtrada por la función {cod_funcion}")
    return errores

def calibrar(anio, mes, nivel, depto):
    """Elige el formato de URL que trae la tabla y verifica el código de la genérica 2.6 que usa el MEF."""
    global FORMATO, GENERICA_INVERSION
    print("🔗 Calibrando formato de URL…")
    for f in FORMATOS:
        p, err = consultar(anio, mes, nivel, depto, [], CLAVE_GENERICA, formato=f, reintentos=2)
        n = len(p.filas) if p else 0
        print(f"   · {f['nombre']:<15} -> {err or f'{n} filas de detalle'}")
        if p and n and not validar(p, nivel, depto, None):
            FORMATO = f
            for v, et, _ in p.filas:
                if re.search(r"activos no financieros", normalizar(et)) and v != GENERICA_INVERSION:
                    print(f"   genérica 2.6: el MEF usa '{v}' (antes '{GENERICA_INVERSION}')"); GENERICA_INVERSION = v
            return
    raise RuntimeError("Consulta Amigable no devolvió la tabla de detalle con ningún formato de URL "
                       "(¿bloqueo del MEF? prueba más tarde o revisa FORMATOS / PARAMS_EXTRA).")

def proyectos_de(anio, mes, nivel, depto, cod_funcion):
    """-> (proyectos [(código, nombre, devengado)], total de la función según el MEF, error, aviso)."""
    p, err = consultar(anio, mes, nivel, depto, [(CLAVE_GENERICA, GENERICA_INVERSION), (CLAVE_FUNCION, cod_funcion)], CLAVE_PROYECTO)
    if err:
        return [], None, err, None
    e = validar(p, nivel, depto, cod_funcion)
    if e:
        return [], None, "; ".join(e), None
    total = p.filtros[-1][1] if p.filtros else None
    proyectos = []
    for valor, et, d in p.filas:
        m = _CODIGO.match(et)
        proyectos.append((m.group(1) if m else valor, re.sub(r"^[\w\-\.]+\s*:\s*", "", et).strip(), d))
    aviso = None
    if total is not None and abs(sum(x[2] for x in proyectos) - total) > max(1, len(proyectos)):
        aviso = f"la suma de proyectos ({sum(x[2] for x in proyectos):,.0f}) no cuadra con el total de la función ({total:,.0f}): ¿lista paginada?"
    return proyectos, total, None, aviso

In [ ]:
# ═══ BLOQUE 5 · EXTRACCIÓN DE PROYECTOS ═══
if not plan:
    raise RuntimeError("No hay departamentos/niveles con datos en los dos meses comparados (ver avisos del Bloque 3).")
p0 = plan[0]
calibrar(p0["fecha"].year, str(p0["fecha"].month), p0["nivel"]["codigo"], p0["cod_depto"])

errores = []
for p in plan:
    anio, mes, nivel = p["fecha"].year, str(p["fecha"].month), p["nivel"]["codigo"]
    for f in p["funciones"]:
        cods = ["15", "16"] if AGRUPAR_COMUNICACIONES_EN_TRANSPORTE and f["cod"] == "15" else [f["cod"]]
        todos, total_mef = [], 0.0
        for c in cods:
            ctx = f"{p['departamento']} · {p['nivel']['nombre']} · {etq_mes(p['fecha'])} · función {c}"
            proyectos, total, err, aviso = proyectos_de(anio, mes, nivel, p["cod_depto"], c)
            if err:
                if c == "16":           # sin Comunicaciones en ese mes: la función 15 basta
                    continue
                errores.append(f"{ctx}: {err}"); continue
            if aviso:
                avisos.append(f"{ctx}: {aviso}")
            todos += proyectos; total_mef += total or 0.0
        todos.sort(key=lambda x: x[2], reverse=True)
        f["proyectos"] = todos[:TOP_PROYECTOS]
        f["devengado_funcion"] = total_mef
        f["n_proyectos"] = len(todos)
    print(f"✔ {p['departamento']} · {p['nivel']['nombre']} ({etq_mes(p['fecha'])})")

if errores:
    print("\n❌ Consultas con error (esas funciones quedan sin proyectos):")
    for e in errores:
        print("  ·", e)

In [ ]:
# ═══ BLOQUE 6 · EXCEL Y SUBIDA A DRIVE ═══
F_TIT = Font(name="Arial", size=9, bold=True); F_NOR = Font(name="Arial", size=9); F_SUB = Font(name="Arial", size=9, italic=True)
F_SUBE = Font(name="Arial", size=9, color="1E7B34"); F_BAJA = Font(name="Arial", size=9, color="C00000")
F_SUBE_B = Font(name="Arial", size=9, bold=True, color="1E7B34"); F_BAJA_B = Font(name="Arial", size=9, bold=True, color="C00000")
FILL_H = PatternFill("solid", fgColor="D9E1F2"); FILL_T = PatternFill("solid", fgColor="F2F2F2")

def color(v, negrita=False):
    if negrita:
        return F_SUBE_B if v > 0 else F_BAJA_B if v < 0 else F_TIT
    return F_SUBE if v > 0 else F_BAJA if v < 0 else F_NOR

wb = Workbook(); wb.remove(wb.active)
detalle = []
ANCHOS = [4, 12, 90, 16, 12]                       # # | Código | Proyecto | Devengado S/ | % de la función
for cod, nombre in DEPARTAMENTOS.items():
    ws = wb.create_sheet(nombre[:31])
    ws["A1"] = f"{nombre}: PROYECTOS DETRÁS DE LAS VARIACIONES DE LA INVERSIÓN (GENÉRICA 2.6) · {etq_mes(u)} vs {etq_mes(u_ant)}"; ws["A1"].font = F_TIT
    ws["A2"] = (f"Soles corrientes · total del nivel positivo: {TOP_PROYECTOS} proyectos con mayor devengado de {etq_mes(u)} en las {TOP_FUNCIONES} "
                f"funciones que más crecieron · negativo: de {etq_mes(u_ant)} en las {TOP_FUNCIONES} que más cayeron"); ws["A2"].font = F_SUB
    for j, w in enumerate(ANCHOS, start=1):
        ws.column_dimensions[L(j)].width = w
    r = 4
    for p in [x for x in plan if x["cod_depto"] == cod]:
        signo = "▲ aumento" if p["positivo"] else "▼ caída"
        ws.cell(r, 1, f"{p['nivel']['nombre']} · {etq_mes(u)} vs {etq_mes(u_ant)}: {p['diferencia']:+,.0f} ({signo}) "
                      f"→ proyectos de {etq_mes(p['fecha'])}").font = F_TIT
        for cc in range(1, len(ANCHOS) + 1):
            ws.cell(r, cc).fill = FILL_H
        r += 1
        for j, (t, v) in enumerate([(f"Total {etq_mes(u_ant)}", p["total_anterior"]), (f"Total {etq_mes(u)}", p["total_actual"]),
                                    ("Diferencia S/", p["diferencia"])]):
            ws.cell(r + j, 3, t).font = F_NOR
            c = ws.cell(r + j, 4, v); c.number_format = "#,##0"; c.font = color(v, True) if j == 2 else F_NOR
        r += 4
        for i, f in enumerate(p["funciones"], start=1):
            ws.cell(r, 1, i).font = F_TIT
            ws.cell(r, 2, f"{f['nombre']} · {etq_mes(u_ant)}: {f['anterior']:,.0f} · {etq_mes(u)}: {f['actual']:,.0f} · "
                          f"Diferencia S/ {f['diferencia']:+,.0f}").font = F_TIT
            r += 1
            for j, t in enumerate(["#", "Código", f"Proyecto ({etq_mes(p['fecha'])})", "Devengado S/", "% función"], start=1):
                c = ws.cell(r, j, t); c.font = F_TIT; c.fill = FILL_T
                c.alignment = Alignment(horizontal="left" if j == 3 else "center")
            r += 1
            proyectos = f.get("proyectos", [])
            if "proyectos" not in f:
                ws.cell(r, 3, "No se pudo consultar esta función (ver errores al correr el notebook)").font = F_BAJA
                r += 1
            elif not proyectos:
                ws.cell(r, 3, f"Sin proyectos con devengado en {etq_mes(p['fecha'])}").font = F_SUB
                r += 1
            for k, (codigo, proyecto, d) in enumerate(proyectos, start=1):
                part = d / f["devengado_funcion"] * 100 if f["devengado_funcion"] else None
                for j, v in enumerate([k, codigo, proyecto, d, part if part is not None else "n.a."], start=1):
                    c = ws.cell(r, j, v); c.font = F_NOR
                    c.number_format = "#,##0" if j == 4 else "0.0" if j == 5 else "General"
                    if j == 3:
                        c.alignment = Alignment(wrap_text=True, vertical="top")
                detalle.append({"DEPARTAMENTO": nombre, "NIVEL": p["nivel"]["nombre"], "DIFERENCIA NIVEL S/": p["diferencia"],
                                "AÑO CONSULTADO": etq_mes(p["fecha"]), "# FUNCIÓN": i, "FUNCIÓN": f["nombre"],
                                "DIFERENCIA FUNCIÓN S/": f["diferencia"], "# PROYECTO": k, "CÓDIGO": codigo, "PROYECTO": proyecto,
                                "DEVENGADO S/": d, "% DE LA FUNCIÓN": part, "PROYECTOS EN LA FUNCIÓN": f["n_proyectos"]})
                r += 1
            r += 1
        r += 1
    ws.freeze_panes = "A4"

ws = wb.create_sheet("DETALLE")
cols = ["DEPARTAMENTO", "NIVEL", "DIFERENCIA NIVEL S/", "AÑO CONSULTADO", "# FUNCIÓN", "FUNCIÓN", "DIFERENCIA FUNCIÓN S/",
        "# PROYECTO", "CÓDIGO", "PROYECTO", "DEVENGADO S/", "% DE LA FUNCIÓN", "PROYECTOS EN LA FUNCIÓN"]
for j, (t, w) in enumerate(zip(cols, [12, 20, 16, 14, 9, 40, 16, 10, 10, 90, 14, 12, 12]), start=1):
    c = ws.cell(1, j, t); c.font = F_TIT; c.fill = FILL_H
    ws.column_dimensions[L(j)].width = w
for i, fila in enumerate(detalle, start=2):
    for j, t in enumerate(cols, start=1):
        v = fila[t]
        c = ws.cell(i, j, None if isinstance(v, float) and np.isnan(v) else v); c.font = F_NOR
        if t in ("DIFERENCIA NIVEL S/", "DIFERENCIA FUNCIÓN S/", "DEVENGADO S/"):
            c.number_format = "#,##0"
        elif t == "% DE LA FUNCIÓN":
            c.number_format = "0.0"
ws.auto_filter.ref = f"A1:{L(len(cols))}{max(len(detalle), 1) + 1}"
ws.freeze_panes = "A2"

nombre_arch = f"{PROYECTOS_NOMBRE} {u:%Y-%m}.xlsx"
ruta_arch = os.path.join("/content", nombre_arch)
wb.save(ruta_arch)

def subir_xlsx_a_drive(ruta_local, folder_id, max_reintentos=5):
    """Crea o actualiza (por nombre) el archivo en la carpeta de Drive. Requiere permiso de Editor."""
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.errors import HttpError
    from googleapiclient.http import MediaFileUpload
    auth.authenticate_user()
    servicio = build("drive", "v3", cache_discovery=False)
    carpeta = servicio.files().get(fileId=folder_id, fields="name,mimeType,capabilities(canAddChildren)",
                                   supportsAllDrives=True).execute()
    if carpeta.get("mimeType") != "application/vnd.google-apps.folder":
        raise ValueError(f"El ID {folder_id} no corresponde a una carpeta de Drive.")
    if not carpeta.get("capabilities", {}).get("canAddChildren", False):
        raise PermissionError(f"Sin permiso de Editor en '{carpeta['name']}'.")
    nombre = os.path.basename(ruta_local)
    nombre_q = nombre.replace("\\", "\\\\").replace("'", "\\'")
    mime = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    for intento in range(1, max_reintentos + 1):
        try:
            existentes = servicio.files().list(q=f"name = '{nombre_q}' and '{folder_id}' in parents and trashed = false",
                                               fields="files(id)", supportsAllDrives=True,
                                               includeItemsFromAllDrives=True).execute().get("files", [])
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if existentes:
                servicio.files().update(fileId=existentes[0]["id"], media_body=media, supportsAllDrives=True).execute()
                return f"actualizado en '{carpeta['name']}'"
            servicio.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                    fields="id", supportsAllDrives=True).execute()
            return f"creado en '{carpeta['name']}'"
        except HttpError as e:
            if e.resp.status not in {403, 429, 500, 502, 503, 504} or intento == max_reintentos:
                raise
            time.sleep(2 ** intento)

try:
    print(f"☁️  {nombre_arch}: {subir_xlsx_a_drive(ruta_arch, PROYECTOS_FOLDER_ID)}")
except Exception as e:
    import shutil
    os.makedirs(RUTA_SALIDA, exist_ok=True)
    respaldo = os.path.join(RUTA_SALIDA, nombre_arch)
    shutil.copy(ruta_arch, respaldo)
    print(f"❌ No se pudo subir a la carpeta {PROYECTOS_FOLDER_ID}: {e}\n   Copia guardada en {respaldo}")

for a in avisos:
    print("⚠️", a)
if detalle:
    print(f"\nProyecto #1 de cada función ({etq_mes(u)} vs {etq_mes(u_ant)}):")
    det = pd.DataFrame(detalle)
    display(det[det["# PROYECTO"] == 1][["DEPARTAMENTO", "NIVEL", "AÑO CONSULTADO", "FUNCIÓN", "CÓDIGO", "PROYECTO", "DEVENGADO S/"]]
            .reset_index(drop=True))